# 1.2 昇腾 950 架构：为 LLM 推理重构

昇腾 950 系列（950PR / 950DT）采用**第三代 DaVinciCore**，面向大语言模型推理场景全面重构：新增 SIMT 执行模式、低精度格式（HiF8/MXFP8/MXFP4）、CV 融合数据通路……而这一切，都直接服务于本课程的主角——**FlashAttention**：

> 根据《昇腾 950 NPU 架构白皮书》，950 针对 FlashAttention 做了深度硬件优化，**单核性能相比上一代提升 1.5 ~ 2 倍**。

本节自顶向下：芯片整体（Chiplet）→ 装备清单 → 架构演进（从耦合到分离）→ AI Core 子系统 → Vector Core 双模执行 → CV 融合通路 → GM 访问的真实旅程，最后对照前代总结差异。

## 1. 芯片整体：Chiplet 统一内存架构

950PR 单芯片内部合封：**2 个 AI Die + 2 个 IO Die + 8 个高速片上内存模块**，Die 间以 D2D 高速互联。这带来的三大收益是**可扩展**（良率与成本）、**统一编程**（多 Die 对程序员透明）与**大容量片上内存**。

<img src="./images/950_chiplet_arch.png" width="620">

> **看图要点**：AI Die 承载计算（AI Core 子系统），IO Die 承载对外接口（UB 灵衢互联、PCIe 等），8 个片上内存模块提供 128GB 统一内存。相比传统“显存挂在一侧”的设计，Chiplet UMA 让容量与带宽可以随模块数扩展。

## 2. 950PR 技术规格：装备清单

（摘自昇腾官网，32/28 表示两种规格）

| 规格项 | Ascend 950PR |
|--|--|
| Cube Core 数量 | 32 / 28 |
| Vector Core 数量 | 64 / 56 |
| MXFP4 算力 | 1784 / 1561 TFLOPS |
| HiF8 / MXFP8 / FP8 算力 | 919 / 804 TFLOPS |
| INT8 算力 | 919 / 804 TOPS |
| BF16 / FP16 算力 | 486 / 425 TFLOPS |
| TF32 算力 | 243 / 212 TFLOPS |
| Vector FP16/BF16 算力 | 54 / 47 TFLOPS |
| 内存容量 | 128 / 112 GB |
| 内存带宽 | 1.6 / 1.4 TB/s |
| L2 Cache | 128 / 112 MB，512B Cache Line |
| 互联 | UB 灵衢 2.0（2TB/s）、PCIe 5.0 x16 |

**两个对 FA 至关重要的观察**：

1. **精度阶梯**：MXFP4 : FP8 : FP16 ≈ 4 : 2 : 1——低精度即高算力，这是第6章全量化 FA 的硬件依据；
2. **算力配比**：Cube FP16 432T vs Vector 2×27T ≈ **8 : 1**——FA 中夹在两次矩阵乘之间的 Softmax/Rescale 段是 Vector 活，配比失衡意味着**Vector 容易成为 FA 的瓶颈**，这是第7~9章性能优化的核心矛盾。

## 3. 架构演进：从耦合到分离

达芬奇架构按 Cube 与 Vector 计算单元是否同核部署，分为两种形态：

| 形态 | 代表芯片 | 结构 | 执行/内存单元 |
|--|--|--|--|
| 耦合架构 | 310 / 910A | Cube、Vector、Scalar 同在一个核 | 6 个异步执行单元：3 计算（Cube/Vector/Scalar）+ 3 搬运（MTE1/2/3）；6 级内存（GM/L1/UB/L0A/L0B/L0C） |
| 分离架构 | 910B 起（含 910D/950） | 拆成 AIC 与 AIV 两个独立的核 | AIC：MTE1/MTE2/MTE3/Cube/Scalar 5 个执行单元 + 7 个内存单元；AIV：MTE2/MTE3/Vector/Scalar 4 个执行单元 + 2 个内存单元（GM、UB） |

> **分离的动机**：矩阵计算与向量计算解耦，各自拥有 Scalar、独立加载自己的代码段，由系统软件统一调度配合——两类计算都能跑满，互不拖累。AIC 与 AIV 之间通过 Global Memory 传递数据；分离架构还新增两个小仓库：**BT Buffer**（BiasTable，存 Bias）与 **FP Buffer**（Fixpipe Parameter，存量化参数、ReLU 参数等）。

### 3.1 三代规格对比：910B → 910D → 950

| 规格项 | 910B | 910D | 950 |
|--|--|--|--|
| AIC / AIV 核数 | 24 / 48 | 32 / 64 | 32 / 28 或 64 / 56（两档规格） |
| Cube FP16 算力 | 353.9T | 432.5T | 486 / 425T |
| UB | 192KB | 256KB | 256KB 级 |
| L0C | 128KB | 256KB | 256KB |
| L1 | 512KB | 512KB | MB 级（子系统共享） |
| L2 | 192MB，512B Line | 128MB，128B×4 Sector | 128 / 112MB，512B Line |
| 主频 | 1.8GHz | 1.65GHz | — |
| TLB 存放位置 | GM | L2 Cache | — |

> 两个演进信号：**① 容量**——UB 与 L0C 翻倍意味着“基本块”可以更大，单次搬运/计算的数据更多，对带宽的总需求更小；**② 地址翻译**——910D 把 TLB 搬进 L2，翻译页表的代价大幅下降（910B 的 TLB miss 要去 GM 换页表，见第 7 节）。

### 3.2 伏笔：UB 的 Bank 组织

UB 并非黑箱一块，它由多组 **Bank** 拼成，同一 Bank 同一拍只允许 1 读或 1 写：

- **910B（1971）**：16 个 Bank Group × 每 Group 3 个 Bank（共 192KB）；
- **910D（David）**：8 个 Bank Group × 每 Group 2 个 Bank（共 256KB）。

若向量指令的操作数恰好落在同一 Bank 的冲突地址上，访问被串行化——即 **Bank 冲突**，是性能调优（第7~9章）的检查项之一。现在只需记住：**UB 有内部结构，数据的地址排布会影响有效带宽**。

## 4. AI Core 子系统微架构

950 的 AI 子系统由 **1 个 AIC（AI Cube 核）+ 2 个 AIV（AI Vector 核）** 组成一个子系统，全芯片共 36 个子系统。

<img src="./images/950_ai_core_microarch.png" width="640">

> **看图要点**：一 Cube 两 Vector 的对称设计是 950 的基本计算单元。相比前代，第三代 DaVinciCore 有六处升级：L0C→UB 直通数据通路、更大的 UB 容量、SS Buffer、SIMT 执行模式、RegBase 编程（寄存器级寻址），以及张量核心浮点峰值算力提升至前代 4 倍。Cube 与 Vector 之间的协作方式，将在下一张图（CV 融合）中揭晓。

> **演进铺垫**：这些升级并非凭空出现。上一代 910D 相对 910B 已经打通了 **L1 ↔ UB 双向通路**与 **L0C → UB 直连**（各 256B/周期），新增 ssbuf 传递 UB→L1、L0C→UB 的目的地址，同时把 UB 上 Vector 计算从 memory base 改为 **reg base**——中间结果留在寄存器里直接算，不必回写 UB。950 的 CV 融合与 RegBase 编程，正是把这条演进路线推向成熟。

## 5. Vector Core：SIMD + SIMT 双模执行

950 的 Vector Core 支持**两种执行模式**，这是与前代最大的编程模型差异：

| 模式 | 一句话 | 适合 |
|--|--|--|
| **SIMD**（单指令多数据） | 一条指令作用于一段向量，经典昇腾模式 | 规整的大块向量计算（FA 的 Softmax） |
| **SIMT**（单指令多线程） | 类 CUDA 的线程级并行，每线程独立寄存器与分支 | 不规则、分支密集的逻辑 |

<img src="./images/vector_core_arch.png" width="640">

> **看图要点**：左侧是 SIMD/SIMT 共享的基础设施；右上为 SIMD 模式——乱序发射引擎 + 多 Bank UB + 向量寄存器堆，支持**双发射**（单 Vector Core 128 FP16 MAC/周期，双发射 256）；右下为 SIMT 模式——Warp Scheduler 顺序发射、线程分支可发散。950 还支持两者**混合编程**。对 FA 而言，规整的 Online Softmax 用 SIMD 即可打满，SIMT 为复杂掩码等场景提供灵活性。

## 6. CV 融合：为 FA 而生的数据通路

前代芯片上，Cube 的结果（L0C）要回到 GM 再进 UB 才能 Vector 处理；950 打通了 **L0C → UB 直连通路**：

<img src="./images/cube_vector_fusion.png" width="640">

> **看图要点**：对 FA 这意味着 **QK^T 的结果 S 不必绕道片外，直接进 UB 做 Softmax，Softmax 结果直接可参与下一次矩阵乘**——「MatMul + Softmax 片上一次完成」。配套的还有**随路处理**：L0C → UB 搬运时可顺带完成 FP32 → 低精度量化与 NZ → ND 排布转换，零额外开销。第6章量化 FA 会直接受益于此。

这就是 950 对 FA 单核性能提升 1.5~2 倍的三板斧之一（另两板斧：Vector 双发射缓解 8:1 配比矛盾、超大 UB 容纳更多 KV 块）。

## 7. GM 的真身：HBM、L2 与 SMMU/MATA

程序员眼中的 GM，落在芯片上是 **HBM + L2 Cache** 两个实体。AI Core 一次 GM 读访问的真实旅程：

1. AI 核发起访存指令，**SMMU**（System MMU，每个 AI 核各一个）将虚拟地址翻译为物理地址；
2. **MATA/HA**（Home Agent，每个 HA 负责一块 HBM）接收请求，先查 **L2 Cache**——命中则直接把数据送回 AI Core；
3. 未命中则访问 HBM，把数据加载进 L2，再送往 AI Core。

三个性能要点（第7章性能优化会反复遇到它们）：

- **TLB miss 很贵**：地址翻译的页表项不在 TLB 时要去 DDR 取页表，最坏还要落到磁盘。大跨度、不规整的访存模式最容易触发——Tiling 设计应尽量让地址连续、规整；
- **Cache Line / Sector 对齐**：L2 按 Cache Line 整行加载（512B；910D 起引入 128B Sector），访存地址对齐能显著减少实际搬运量；
- **多核同址竞争**：多核同时访问同一 L2/HBM 地址会互相抢占带宽——FA 多核切分（第9章）要避开热点地址。

## 8. 950 与 A2/A3 对比：一张迁移视角的表

| 维度 | Atlas A2/A3（910B/C） | Ascend 950 |
|--|--|--|
| 架构代际 | 第二代 DaVinciCore | 第三代 DaVinciCore |
| 子系统结构 | AIC + AIV | 1 AIC + 2 AIV（更均衡） |
| Vector 执行 | SIMD | SIMD（双发射）+ SIMT + 混合 |
| 低精度格式 | FP16/BF16/INT8 | 新增 HiF8/MXFP8/FP8/MXFP4 |
| CV 数据通路 | L0C → GM → UB（绕片外） | **L0C → UB 直连 + 随路量化** |
| 搬运 | 传统 MTE 体系 | 传统 MTE + NDDMA（5 维重排） |
| 调度 | Host 下发为主 | STARS2.0 片上自主调度 |

> 对本课程最重要的一行是 **CV 数据通路**：同一个 FA Kernel，在 950 上能获得显著更低的片内往返开销——第3篇写 Kernel、第5篇做优化时都会反复回到这张表。

## 小结

| 关键词 | 一句话 |
|--|--|
| Chiplet UMA | 2 AI Die + 2 IO Die + 8 内存模块，128GB 统一片上内存 |
| 装备清单 | FP16 486T、MXFP4 1784T、内存 1.6TB/s |
| 双模执行 | SIMD 打满规整向量，SIMT 处理不规则分支 |
| CV 融合 | L0C→UB 直连，MatMul+Softmax 片上一次完成 |
| FA 专属 | 950 对 FlashAttention 单核性能 +1.5~2 倍 |
| 分离架构 | 310 耦合起步，910B 起拆成 AIC/AIV 各自独立 |
| GM 真身 | HBM + L2，SMMU 翻译地址、MATA/HA 查 L2 再到 HBM |

至此装备清点完毕。接下来进入“六块积木”环节：[2.1 算子数据流](01.03_operator_dataflow.ipynb)。

## 课后练习

1.（单选）950PR 的 Cube FP16 与 Vector FP16 算力配比约为？（A. 1:1　B. 4:1　C. 8:1　D. 16:1）

2.（判断）950 的 CV 融合通路使得 QK^T 的结果可以不经过 GM 直接进入 UB 参与 Softmax 计算。

3.（单选）下列哪项是 950 相对前代新增、且与 FA 最相关的特性？（A. PCIe 5.0　B. L0C→UB 直连与随路量化　C. 片上内存 128GB　D. HCCS 互联）

> 完成后查看 [answer/01.02_answer.txt](answer/01.02_answer.txt) 核对答案。